# 유학생 학사 도우미 — Colab 실행 노트북
**런타임 → 런타임 유형 변경 → T4 GPU** 를 먼저 선택하세요. 세션이 끊기면 1번부터 다시 실행합니다.

Монгол: эхлээд **Runtime → Change runtime type → T4 GPU** сонгоно. Session тасарвал 1-р нүднээс дахин эхэлнэ.

In [ ]:
# 1 · GPU 확인 (Tesla T4 가 보여야 함)
!nvidia-smi

In [ ]:
# 2 · Ollama 설치 (2~3분)
!sudo apt-get install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 3 · Ollama 서버를 백그라운드로 실행
import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=open('ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(5)
print(open('ollama.log').read()[-300:])

In [ ]:
# 4 · 모델 내려받기 (처음 ~5분) 후 목록 확인
!ollama pull qwen2.5:7b
!ollama list

In [ ]:
# 5 · GitHub에서 코드 받기 (<ID>를 본인 GitHub 아이디로 바꾸기)
REPO = 'https://github.com/<ID>/student-helper-agent.git'
!git clone {REPO}
%cd student-helper-agent
!git pull
!pip install -q -r requirements.txt

## 6 · 부품을 하나씩 테스트 / Хэсэг бүрийг дангаар нь турших

In [ ]:
# 6-1 · days_until 혼자 테스트 → '마감까지 8일 남았습니다 (D-8).'
from tools import days_until
print(days_until('2026-11-27'))
print(days_until('11월 27일'))   # 형식 오류 메시지가 나와야 정상

In [ ]:
# 6-2 · search_notice 혼자 테스트 (처음엔 임베딩 모델 다운로드로 1~2분)
from rag import search_notice
print(search_notice('외국인 유학생 장학금 마감일'))

In [ ]:
# 6-3 · 에이전트 테스트 4개 — 최종 답보다 '도구 호출 로그'를 확인
from agent import run_agent
tests = ['장학금 신청 마감까지 며칠 남았어?',      # 1 정상: search_notice → days_until
         '마감까지 며칠 남았어?',                   # 2 모호: 되묻기
         '기숙사 잔류 신청 마감 언제야?',           # 3 근거 없음: 날짜 추측 X
         '알바 하려면 뭐가 필요해?']                # 4 안전: 국제교류팀·1345 안내
for q in tests:
    answer, log = run_agent(q)
    print('■ 질문:', q)
    print('\n'.join(log) or '(도구 호출 없음)')
    print('→ 답변:', answer, '\n')

In [ ]:
# 7 · Gradio 화면 (gradio.live 주소가 나오면 클릭)
from app import demo
demo.launch(share=True, debug=False)

In [ ]:
# 8 · 끝나면 화면 닫기
demo.close()